<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/bonus_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama ollama pypdf

In [ ]:
!nohup ollama serve &

In [ ]:
%%bash
ollama pull embeddinggemma:300m

In [2]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)

## Semantic search

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("acmecorp-employee-handbook.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=20, add_start_index=True
)
all_splits = text_splitter.split_documents(documents=documents)
len(all_splits)

13

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding = OllamaEmbeddings(model="embeddinggemma:300m")

vector_store = InMemoryVectorStore(embedding=embedding)
ids = vector_store.add_documents(documents=all_splits)

In [5]:
results = vector_store.similarity_search(query="""
How many days of vacation does an employee get in their first year?
""")

results[0:3]

[Document(id='bc480663-433c-495b-bd61-f6704d7c3b6f', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'start_index': 897}, page_content='Full■time employees accrue PTO according to the following schedule: \x7f 0–1 years of service: 10 days\nper year (0.833 days per month) \x7f 1–3 years of service: 15 days per year (1.25 days per month) \x7f 3+'),
 Document(id='5de95dd6-992b-49b0-a1d4-8b63c07a2e26', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(

## RAG Agent

In [6]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    base_url="https://ollama.com",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [7]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """
    Search the employee handbook for information
    """
    results = vector_store.similarity_search(query=query, k=3)
    return "\n\n".join([doc.page_content for doc in results])

In [8]:
from langchain.agents import create_agent

system_prompt = """
You are an assistant that answers questions
ONLY by using the search_handbook tool.

When a user asks a question:
1. ALWAYS call the search_handbook tool first.
2. Use the retrieved content to answer.
3. If the answer is not found, say you don't know.

Do not answer from your own knowledge.
"""

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt=system_prompt
)

In [9]:
import time
from langchain.messages import HumanMessage

start_time = time.time()

messages = [HumanMessage(content="""
How many days of vacation does an employee get in their first year?
""")]
response = agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


How many days of vacation does an employee get in their first year?

================================== Ai Message ==================================
Tool Calls:
  search_handbook (c70bf064-d1e4-407c-9a7c-d09db7c8b357)
 Call ID: c70bf064-d1e4-407c-9a7c-d09db7c8b357
  Args:
    query: vacation days first year
================================= Tool Message =================================
Name: search_handbook

Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+

years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to

an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceed